# Historical Analysis of Observed Weather At KJFK

In this example, we will perform a comprehensive analysis of the past weather at 
John F. Kennedy International Airport in NYC (KJFK).

We will use xmacis2py to perform the following:

1) Download Data starting at January 1st, 1950.
2) Calculate Daily Climatological Normals based on this longer climatology.
3) Performing Analog Queries for the top 3 El Nino and top 3 La Nina December-January-February (DJF) periods.
4) Calculating Weighted Means and Percentiles for Average Temperature Departure on our filtered analog years.

## Imports

In [1]:
from xmacis2py import get_single_station_acis_data, analysis
from datetime import datetime, timedelta

In [2]:
yesterday = datetime.now() - timedelta(days=1)

In [3]:
yesterday

datetime.datetime(2026, 4, 24, 11, 52, 5, 497963)

In [4]:
df = get_single_station_acis_data('kjfk',
                                  start_date='1950-01-01',
                                  end_date=yesterday)

In [5]:
df

,Date,Maximum Temperature,Minimum Temperature,Average Temperature,Average Temperature Departure,Heating Degree Days,Cooling Degree Days,Precipitation,Snowfall,Snow Depth,Growing Degree Days
0,1950-01-01,41.0,31.0,36.0,1.4,29.0,0.0,0.001,0.0,0.0,0.0
1,1950-01-02,49.0,37.0,43.0,8.6,22.0,0.0,0.001,0.0,0.0,0.0
2,1950-01-03,51.0,45.0,48.0,13.8,17.0,0.0,0.100,0.0,0.0,0.0
3,1950-01-04,62.0,47.0,54.5,20.5,10.0,0.0,0.000,0.0,0.0,5.0
4,1950-01-05,62.0,47.0,54.5,20.6,10.0,0.0,0.000,0.0,0.0,5.0
...,...,...,...,...,...,...,...,...,...,...,...
27868,2026-04-20,54.0,36.0,45.0,-7.5,20.0,0.0,0.000,0.0,0.0,0.0
27869,2026-04-21,49.0,34.0,41.5,-11.3,23.0,0.0,0.001,0.0,NaN,0.0
27870,2026-04-22,52.0,45.0,48.5,-4.6,16.0,0.0,0.050,0.0,0.0,0.0
27871,2026-04-23,75.0,42.0,58.5,5.0,6.0,0.0,0.000,0.0,0.0,9.0


## Calculating Our Daily Climate Normals

***def calculate_daily_normals(station,
                            df=None,
                            input_path=None,
                            start_date=None,
                            end_date=None,
                            to_csv=False,
                            output_path=f"XMACIS2 DAILY NORMALS",
                            return_pandas_df=True):***

    This function calculates daily climatological normals for a user-specified period.
    
    This function is useful for those who do not want the day to day fluctuations smoothed out
    as xmACIS2 smooths out the normals (the ones downloaded from the server via get_single_station_climate_normals()).
    
    This is also useful for creating daily climatology normals for a custom period. (i.e. a 50-year climatology)
    
    Required Arguments: 
    
    1) station (String) - The 4-letter ID of the ACIS2 station.
    
    Optional Arguments
    
    1) df (Pandas.DataFrame) - Default=None. If the user is passing in a dataframe (df) without reading in the data from a CSV
        file, set df=df.
        
    2) input_path (String) - Default=None. If the user is reading in data from a CSV file, enter the full path to the
        CSV file.
        
    3) start_date (String or Datetime) - Default=None. For users who want specific start and end dates for their analysis,
        they can either be passed in as a string in the format of 'YYYY-mm-dd' or as a datetime object.
        
    4) end_date (String or Datetime) - Default=None. For users who want specific start and end dates for their analysis,
        they can either be passed in as a string in the format of 'YYYY-mm-dd' or as a datetime object.
        
    5) to_csv (Boolean) - Default=False. When set to True, a CSV file of the data will be created and saved to the user specified path.
    
    6) output_path (String) - Default="XMACIS2 DAILY NORMALS". The output directory hosting the CSV file (only needed if to_csv=True).
    
    7) return_pandas_df (Boolean) - Default=True. When set to True, a pandas.DataFrame is returned.
        To only download CSV files and not return a pandas.DataFrame for each file set to False. 
        
    Returns
    -------
    
    A Pandas.DataFrame of daily climatological normals for a custom period.     

In [6]:
daily_normals = analysis.calculate_daily_normals('kjfk',
                                                 df=df)

In [7]:
daily_normals

,Day Of Year,Maximum Temperature,Minimum Temperature,Average Temperature,Heating Degree Days,Cooling Degree Days,Growing Degree Days,Precipitation,Snowfall,Snow Depth,Date
0,1,41.818182,29.012987,35.415584,29.376623,0.0,0.129870,0.139465,0.145676,0.426515,01-01
1,2,41.103896,28.532468,34.818182,29.948052,0.0,0.000000,0.089843,0.110397,0.397118,01-02
2,3,40.207792,29.285714,34.746753,30.025974,0.0,0.051948,0.146700,0.132544,0.353029,01-03
3,4,40.883117,28.467532,34.675325,30.116883,0.0,0.142857,0.101101,0.211956,0.426574,01-04
4,5,39.649351,27.636364,33.642857,31.129870,0.0,0.103896,0.059536,0.148647,0.455926,01-05
...,...,...,...,...,...,...,...,...,...,...,...
361,362,41.855263,29.131579,35.493421,29.276316,0.0,0.052632,0.129000,0.164313,0.612030,12-27
362,363,42.263158,30.092105,36.177632,28.605263,0.0,0.092105,0.147313,0.149358,0.462761,12-28
363,364,41.618421,29.723684,35.671053,29.078947,0.0,0.144737,0.073388,0.082164,0.403090,12-29
364,365,40.986842,28.763158,34.875000,29.894737,0.0,0.052632,0.088070,0.219582,0.298612,12-30


## Filtering Analog Years

***def filter_analog_years(station,
                        analogs,
                        df=None,
                        input_path=None,
                        to_csv=False,
                        output_path=f"XMACIS2 ANALOGS",
                        return_pandas_df=True):***

    This function filters for analog periods in the form of month and year. 
    
    This can be useful when wanting to perform an analysis of analog years for seasonal forecasting applications.
    
    Required Arguments: 
    
    1) station (String) - The 4-letter ID of the ACIS2 station.
    
    2) analogs (Tuple List) - A list of tuples that represent the analog periods in the query. 
        Format: [(YYYY 1, mm 1), (YYYY 2, mm2),...., (YYYY n, mm n)]
        Example: Let's query winters 2006, 2016 and 2026
        
        [(2005, 12), (2006, 1), (2006, 2),
        (2015, 12), (2016, 1), (2016, 2),
        (2025, 12), (2026, 1), (2026, 2)]
    
    Optional Arguments
    
    1) df (Pandas.DataFrame) - Default=None. If the user is passing in a dataframe (df) without reading in the data from a CSV
        file, set df=df.
        
    2) input_path (String) - Default=None. If the user is reading in data from a CSV file, enter the full path to the
        CSV file.
    
    3) to_csv (Boolean) - Default=False. When set to True, a CSV file of the data will be created and saved to the user specified path.
    
    4) output_path (String) - Default="XMACIS2 ANALOGS". The output directory hosting the CSV file (only needed if to_csv=True).
    
    5) return_pandas_df (Boolean) - Default=True. When set to True, a pandas.DataFrame is returned.
        To only download CSV files and not return a pandas.DataFrame for each file set to False. 
        
    Returns
    -------
    
    A Pandas.DataFrame of analog years for years 1-n for a period for months 1-n.


***ONI Indices Since 1980***

<img src="https://github.com/edrewitz/xmACIS2Py-Jupyter-Lab-Tutorials/blob/main/diagrams/oni%20index.png?raw=true" width="400" height="300">

***We will Filter Analog Years for the DJF period for the top 3 El Nino Years and top 3 La Nina Years***

***Top 3 El Nino Years (DJF)***: 1) 2016 = 2.6, 2) 1983 = 2.2, 3) 1998 = 2.2.

***Top 3 La Nina Years (DJF)***: 1) 1989 = -1.7, 2) 2000 = -1.7, 3) 2008 = -1.6.

In [8]:
el_nino = analysis.filter_analog_years('kjfk',
                                       [(2015, 12), (2016, 1), (2016, 2),
                                        (1982, 12), (1983, 1), (1983, 2),
                                        (1997, 12), (1998, 1), (1998, 2)],
                                       df=df)

In [9]:
el_nino

,Date,Maximum Temperature,Minimum Temperature,Average Temperature,Average Temperature Departure,Heating Degree Days,Cooling Degree Days,Precipitation,Snowfall,Snow Depth,Growing Degree Days
date,,,,,,,,,,,
1982-12-01,1982-12-01,58.0,43.0,50.5,8.4,14.0,0.0,0.430,0.0,0.0,1.0
1982-12-02,1982-12-02,59.0,53.0,56.0,14.2,9.0,0.0,0.001,0.0,0.0,6.0
1982-12-03,1982-12-03,56.0,53.0,54.5,13.0,10.0,0.0,0.001,0.0,0.0,5.0
1982-12-04,1982-12-04,70.0,55.0,62.5,21.2,2.0,0.0,0.001,0.0,0.0,13.0
1982-12-05,1982-12-05,62.0,56.0,59.0,18.0,6.0,0.0,0.001,0.0,0.0,9.0
...,...,...,...,...,...,...,...,...,...,...,...
2016-02-25,2016-02-25,59.0,38.0,48.5,12.0,16.0,0.0,0.040,0.0,0.0,0.0
2016-02-26,2016-02-26,40.0,29.0,34.5,-2.2,30.0,0.0,0.000,0.0,0.0,0.0
2016-02-27,2016-02-27,40.0,28.0,34.0,-2.9,31.0,0.0,0.000,0.0,0.0,0.0


In [10]:
la_nina = analysis.filter_analog_years('kjfk',
                                       [(1988, 12), (1989, 1), (1989, 2),
                                        (1999, 12), (2000, 1), (2000, 2),
                                        (2007, 12), (2008, 1), (2008, 2)],
                                       df=df)

In [11]:
la_nina

,Date,Maximum Temperature,Minimum Temperature,Average Temperature,Average Temperature Departure,Heating Degree Days,Cooling Degree Days,Precipitation,Snowfall,Snow Depth,Growing Degree Days
date,,,,,,,,,,,
1988-12-01,1988-12-01,48.0,39.0,43.5,1.4,21.0,0.0,0.00,0.000,0.0,0.0
1988-12-02,1988-12-02,42.0,35.0,38.5,-3.3,26.0,0.0,0.00,0.000,0.0,0.0
1988-12-03,1988-12-03,52.0,36.0,44.0,2.5,21.0,0.0,0.00,0.000,0.0,0.0
1988-12-04,1988-12-04,51.0,32.0,41.5,0.2,23.0,0.0,0.00,0.000,0.0,0.0
1988-12-05,1988-12-05,43.0,35.0,39.0,-2.0,26.0,0.0,0.00,0.000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
2008-02-25,2008-02-25,45.0,28.0,36.5,0.0,28.0,0.0,0.00,0.000,2.0,0.0
2008-02-26,2008-02-26,47.0,32.0,39.5,2.8,25.0,0.0,0.12,0.000,1.0,0.0
2008-02-27,2008-02-27,44.0,23.0,33.5,-3.4,31.0,0.0,0.00,0.000,0.0,0.0


## Weighted Means and Percentiles

***def analog_weighted_mean(df,
                  parameter,
                  weights):***

    This function calculates the weighted mean for a given variable.
    
    This is useful when wanting to create weighted means of analogs when
    comparing analog years for seasonal forecasting applications.
    
    Required Arguments:
    
    1) df (Pandas.DataFrame) - The dataframe of ACIS2 data.
    
    2) parameter (String) - The parameter of interest.
    
    3) weights (Float/Integer Array) - An array of numbers (can be both float and int) of the weights applied.
    
    Returns
    -------
    
    The weighted mean of the variable in a Pandas.DataFrame.

***def analog_weighted_percentile(df,
                  parameter,
                  weights,
                  percentile):***

    This function calculates the weighted mean for values of a given percentile.
    
    This is useful when wanting to create weighted means applied to percentile values of analogs when
    comparing analog years for seasonal forecasting applications.
    
    Required Arguments:
    
    1) df (Pandas.DataFrame) - The dataframe of ACIS2 data.
    
    2) parameter (String) - The parameter of interest.
    
    3) weights (Float/Integer Array) - An array of numbers (can be both float and int) of the weights applied.
    
    4) percentile (Float or Int) - A value between 0 and 1. (0.5 = 50th percentile)
    
    Returns
    -------
    
    The weighted mean of a given percentile of the variable in a Pandas.DataFrame. 

***El Nino Years Weights***: 2 for 2016, 1 for 1983 and 0.5 for 1998

***La Nina Years Weights***: 2 for 1989, 1 for 2000 and 0.5 for 2008

***We will use the 25th and 17th percentiles for the El Nino and La Nina Years***

In [12]:
el_nino_mean = analysis.analog_weighted_mean(el_nino, 
                                             'Average Temperature Departure',
                                             [2, 1, 0.5])

In [13]:
el_nino_mean

np.float64(3.788515611372754)

In [14]:
la_nina_mean = analysis.analog_weighted_mean(la_nina, 
                                             'Average Temperature Departure',
                                             [2, 1, 0.5])

In [15]:
la_nina_mean

np.float64(0.28148264433978726)

In [16]:
el_nino_25 = analysis.analog_weighted_percentile(el_nino, 
                                             'Average Temperature Departure',
                                             [2, 1, 0.5],
                                             0.25)

In [17]:
el_nino_25

np.float64(-1.6785714285714288)

In [18]:
el_nino_75 = analysis.analog_weighted_percentile(el_nino, 
                                             'Average Temperature Departure',
                                             [2, 1, 0.5],
                                             0.75)

In [19]:
el_nino_75

np.float64(9.507142857142856)

In [20]:
la_nina_25 = analysis.analog_weighted_percentile(la_nina, 
                                             'Average Temperature Departure',
                                             [2, 1, 0.5],
                                             0.25)

In [21]:
la_nina_25

np.float64(-4.421428571428572)

In [22]:
la_nina_75 = analysis.analog_weighted_percentile(la_nina, 
                                             'Average Temperature Departure',
                                             [2, 1, 0.5],
                                             0.75)

In [23]:
la_nina_75

np.float64(6.4071428571428575)